# TRAINER

In [1]:
import re
import torch
import torch.cuda

import evaluate
from sklearn.model_selection import train_test_split

import pandas as pd
import numpy as np

import nltk
from nltk.tokenize import word_tokenize

from PIL import Image
import torch
from transformers import AutoProcessor, LlavaForConditionalGeneration
from transformers import BitsAndBytesConfig

In [3]:
# Activate GPU
device = "cuda:0" if torch.cuda.is_available() else "cpu"
device

device = torch.device("cuda:0")
device

# MULTIMODAL LABELS
columns = ["uri","text","Additiva","Amore","Anticipazione","Citazione","Contenitore","Contenuto","Disgusto","Divergente","Fiducia","Gioia","Hate speech","Incomprensibile","Ironia","Neutro","Offensivita","Parallela","Paura","Rabbia","Sarcasmo","Sorpresa","Tristezza","img","img_path"]

to_keep=["text","Additiva","Amore","Anticipazione","Disgusto","Divergente","Fiducia","Gioia","Neutro","Parallela","Paura","Rabbia","Sorpresa","Tristezza","img_path"]
target_names = ['Amore','Anticipazione','Disgusto','Fiducia','Gioia','Neutro','Paura','Rabbia','Sorpresa','Tristezza']

# DATASET

In [4]:
# Function to create the 'Type' column
def create_type_column(row):
    if row['Additiva'] == 1:
        return 1
    elif row['Parallela'] == 1:
        return 2
    elif row['Divergente'] == 1:
        return 3
    else:
        return 0

In [5]:
def split_dataset(data, test_size=0.2, eval_size=0.2, random_state=None):
    train_data, test_data = train_test_split(
        data, test_size=test_size, random_state=random_state
    )
    
    adjusted_eval_size = eval_size / (1 - test_size)
    
    train_data, eval_data = train_test_split(
        train_data, test_size=adjusted_eval_size, random_state=random_state
    )
    print(f"Train size: {len(train_data)}")
    print(f"Test size: {len(test_data)}")
    print(f"Eval size: {len(eval_data)}")
    return train_data, test_data, eval_data

# Example usage


In [6]:
df = pd.read_csv('D:\\Users\\andre\\VSCprojects\\erasmus_scripts\datasets\\final_multimodal_hot.csv')
# df = pd.read_csv('/source/andrea/pixtral_code/final_multimodal_hot_CORR.csv')

df = df[to_keep]

df['type'] = df.apply(create_type_column, axis=1)

df.drop(['Additiva', 'Parallela', 'Divergente'], axis=1, inplace=True)

train_df, test_df, eval_df = split_dataset(df, test_size=0.2, eval_size=0.2, random_state=42) 

train_df.reset_index(drop=True, inplace=True)
test_df.reset_index(drop=True, inplace=True)
eval_df.reset_index(drop=True, inplace=True)


not_chosen_columns = ["text", "img_path"]
labels = ['Love', 'Anticipation', 'Disgust', 'Trust', 'Joy', 'Neutral', 'Fear', 'Anger', 'Surprise', 'Sadness']

id2label = {idx:label for idx, label in enumerate(labels)}
label2id = {label:idx for idx, label in enumerate(labels)}

Train size: 387
Test size: 129
Eval size: 129


## Preparazione Immagini e output desiderato

In [7]:
import os
from PIL import Image
import requests
from io import BytesIO
import numpy as np
from torchvision.transforms import ToPILImage


def load_image_from_url_or_path(local_path):
    if os.path.isfile(local_path):
        # Local path
        try:
            image = Image.open(local_path)
            return image
        except Exception as e:
            print(f"Unable to load image from path: {e}")
            return None
    else:
        # URL path
        try:
            response = requests.get(path_or_url)
            response.raise_for_status()
            image = Image.open(BytesIO(response.content))
            return image
        except Exception as e:
            print(f"Unable to load image from URL: {e}")
            return None

def preprocess_labels(examples):
    # Handle a single row
    text = examples["text"]
    image_folder = "D:\\Users\\andre\\VSCprojects\\erasmus_scripts\\images"

    path = os.path.join(image_folder, examples["img_path"])
    examples["img_path"] = path
    # Create labels matrix for a single example
    labels_batch = {k: examples[k] for k in examples.keys() if k in target_names}
    labels_matrix = np.zeros((1, len(target_names)))
    
    for idx, label in enumerate(target_names):
        labels_matrix[0, idx] = labels_batch[label]

    labels_matrix = labels_matrix.astype(int)

    # Add processed fields to the row
    examples["labels"] = labels_matrix[0]
    examples["out"] = f"Emotions Detected: {', '.join([id2label[i] for i, val in enumerate(labels_matrix[0]) if val == 1])}"

    # Load the image
    examples["image"] = load_image_from_url_or_path(path)

    return examples

In [8]:
train_df = train_df.apply(preprocess_labels, axis=1)
test_df = test_df.apply(preprocess_labels, axis=1)
eval_df = eval_df.apply(preprocess_labels, axis=1)

In [9]:
len(train_df)

387

In [10]:
from torch.utils.data import Dataset

class PreloadedDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        image = row["image"]
        if self.transform:
            image = self.transform(image)
        text = row["text"]
        out = row["out"]
        return {"image": image, "text": text, "out": out}

# Example usage
from torchvision import transforms

train_dataset = PreloadedDataset(train_df)
test_dataset = PreloadedDataset(test_df)
eval_dataset = PreloadedDataset(eval_df)

# LOAD MODEL

In [11]:
model_id = "mistral-community/pixtral-12b"

In [11]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

model = LlavaForConditionalGeneration.from_pretrained(model_id,
                                                      torch_dtype=torch.float16,
                                                      device_map=device,
                                                      quantization_config=quantization_config)

Loading checkpoint shards: 100%|█████████████████████████████████████████████████████| 6/6 [00:32<00:00,  5.43s/it]


In [12]:
processor = AutoProcessor.from_pretrained(model_id)
processor.tokenizer.add_special_tokens({'pad_token': '<pad>'})
# model.resize_token_embeddings(len(processor.tokenizer))

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


0

In [17]:
# Processor params
# Image size fixed
processor = AutoProcessor.from_pretrained(model_id)

processor.image_processor.size = {"height": 336, "width": 336}
processor.tokenizer.pad_token = processor.tokenizer.eos_token


# DATACOLLATOR

In [13]:
from PIL import Image

def convert_to_rgb(image):
    if image.mode != "RGB":
        return image.convert("RGB")
    return image

class MyDataCollator:
    
    def __init__(self, processor):
        self.processor = processor
       
    def find_subsequence(self, sequence, subsequence):
            seq_len = len(sequence)
            sub_len = len(subsequence)
            
            for i in range(seq_len - sub_len + 1):
                if torch.equal(sequence[i:i+sub_len], subsequence):
                    return i
            return None
    
    def __call__(self, examples):
        texts=[]
        images=[]
        assistant_res = []

        usr = """You are an expert in emotion classification. Classify the emotions expressed in a Twitter post written in Italian, consisting of an image and a caption-like text. Choose 1-2 emotions from: Love, Anticipation, Disgust, Trust, Joy, Fear, Anger, Surprise, Sadness, Neutral.

Your response must be without any additional text or explanations, and formatted as follows:
Emotions detected: [Emotion(s)]"""

        # print(usr)
        
        for example in examples:
            image = example["image"]
            image = convert_to_rgb(image)
            question = usr + example['text']
            answer = example['out']
            messages = [
                    {
                        "role": "user",
                        "content": [{"type": "text", "content": question},{"type": "image"} ],
                    },
                    {
                        "role": "assistant",
                        "content": answer, 
                    },
                ]
            
            text = self.processor.apply_chat_template(messages)
            
            texts.append(text.strip())
            images.append([image])
            assistant_res.append(answer)
            
            
        batch = self.processor(text = texts, images = images, return_tensors="pt", padding=True)
        
        
        labels = batch["input_ids"].clone()
        
        for i, (input_ids, assistant_response) in enumerate(zip(batch["input_ids"], assistant_res)):
            
            assistant_tokens = self.processor.tokenizer(assistant_response, return_tensors ="pt")['input_ids'].squeeze(0).to(input_ids.dtype)
            start_idx = self.find_subsequence(input_ids, assistant_tokens)
            
            if start_idx is not None:
                labels[i, :start_idx] = -100
                labels[i, start_idx+len(assistant_tokens):] = -100
                
        batch["labels"] = labels
        return batch
        

data_collator = MyDataCollator(processor)

In [14]:
def debug_label_masking(data_collator, dataset, num_samples=2):
    """Debug function to verify label masking in the data collator"""
    
    # Get samples from dataset
    samples = [dataset[i] for i in range(num_samples)]
    
    print("=== DEBUGGING LABEL MASKING ===\n")
    
    # Process through data collator
    batch = data_collator(samples)
    
    for i in range(len(samples)):
        print(f"--- EXAMPLE {i} ---")
        print(f"Original text: {samples[i]['text']}")
        print(f"Expected output: {samples[i]['out']}")
        
        # Get the processed tokens
        input_ids = batch['input_ids'][i]
        labels = batch['labels'][i]
        
        print(f"\nInput IDs shape: {input_ids.shape}")
        print(f"Labels shape: {labels.shape}")
        
        # Decode full input
        full_text = data_collator.processor.tokenizer.decode(input_ids, skip_special_tokens=False)
        print(f"\nFull decoded text:\n{full_text}")
        
        # Find where labels are not -100 (i.e., what the model should learn to predict)
        non_masked_indices = torch.where(labels != -100)[0]
        
        if len(non_masked_indices) > 0:
            # Decode only the non-masked tokens (target response)
            target_tokens = input_ids[non_masked_indices]
            target_text = data_collator.processor.tokenizer.decode(target_tokens, skip_special_tokens=False)
            print(f"\nTarget text (non-masked labels):\n{target_text}")
            
            # Check if this matches the expected assistant response
            expected_response = samples[i]['out']
            assistant_tokens = data_collator.processor.tokenizer(expected_response, return_tensors="pt")['input_ids'].squeeze(0)
            expected_decoded = data_collator.processor.tokenizer.decode(assistant_tokens, skip_special_tokens=False)
            
            print(f"\nExpected assistant response:\n{expected_decoded}")
            
            # Verify masking correctness
            print(f"\nMasking verification:")
            print(f"  Non-masked token count: {len(non_masked_indices)}")
            print(f"  Expected response token count: {len(assistant_tokens)}")
            print(f"  Tokens match: {torch.equal(target_tokens, assistant_tokens)}")
            
        else:
            print("\nWARNING: All labels are masked (-100)!")
        

# Run the debug function
debug_label_masking(data_collator, train_dataset, num_samples=4)

=== DEBUGGING LABEL MASKING ===

--- EXAMPLE 0 ---
Original text: soleil e odiosa e scorretta, ma buttatela fuori
Expected output: Emotions Detected: Disgust

Input IDs shape: torch.Size([4288])
Labels shape: torch.Size([4288])

Full decoded text:
<s>[INST]You are an expert in emotion classification. Classify the emotions expressed in a Twitter post written in Italian, consisting of an image and a caption-like text. Choose 1-2 emotions from: Love, Anticipation, Disgust, Trust, Joy, Fear, Anger, Surprise, Sadness, Neutral.

Your response must be without any additional text or explanations, and formatted as follows:
Emotions detected: [Emotion(s)]soleil e odiosa e scorretta, ma buttatela fuori[IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG][IMG]

## Lora

In [ ]:
from peft import LoraConfig
from peft import get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    use_rslora=True,
    target_modules="all-linear",
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
    
)

model = get_peft_model(model,lora_config).to(device)
model.print_trainable_parameters()
# trainable params: 66,322,432 || all params: 12,749,062,144 || trainable%: 0.5202

In [15]:
model.print_trainable_parameters()

trainable params: 66,322,432 || all params: 12,749,062,144 || trainable%: 0.5202


## Training

In [16]:
from transformers import TrainingArguments, Trainer

epochs = 2
lr = 3e-5
schedule="linear"

run_name = f"pixtral_finetune-{epochs}_epochs_{lr}_lr_emotions_no_correlations_short"

training_args = TrainingArguments(
    num_train_epochs=epochs,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=lr,
    weight_decay=0.01,
    output_dir="/source/andrea/pixtral_code/new_ft_models/emo_no_correlations_short",
    do_eval=True, 
    eval_strategy="steps",
    save_strategy="steps",
    save_steps=10,
    eval_steps=10,
    logging_steps=10, 
    lr_scheduler_type=schedule,
    fp16=True,
    # hub_model_id="lupobricco/prosat_multimodal_pixtral",
    remove_unused_columns=False,
    gradient_checkpointing=True,
    dataloader_pin_memory=False,
    report_to="wandb",
    run_name=run_name
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [17]:
import wandb

wandb.login(key="d0bb3bff433a2edf5c9e7588b0d09e53f4528611")
wandb.init(
        project="pixtral-emit",
        name=run_name
)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/land/.netrc
wandb: Currently logged in as: drusso98 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [18]:
trainer.train()

You are an expert in emotion classification. Classify the emotions expressed in a Twitter post written in Italian, consisting of an image and a caption-like text. Choose 1-2 emotions from: Love, Anticipation, Disgust, Trust, Joy, Fear, Anger, Surprise, Sadness, Neutral.

Your response must be without any additional text or explanations, and formatted as follows:
Emotions detected: [Emotion(s)]
You are an expert in emotion classification. Classify the emotions expressed in a Twitter post written in Italian, consisting of an image and a caption-like text. Choose 1-2 emotions from: Love, Anticipation, Disgust, Trust, Joy, Fear, Anger, Surprise, Sadness, Neutral.

Your response must be without any additional text or explanations, and formatted as follows:
Emotions detected: [Emotion(s)]
You are an expert in emotion classification. Classify the emotions expressed in a Twitter post written in Italian, consisting of an image and a caption-like text. Choose 1-2 emotions from: Love, Anticipatio

/usr/local/lib/python3.10/dist-packages/torch/utils/checkpoint.py:91: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
10,2.542200,0.396642
20,0.258600,0.229903
30,0.263100,0.224920
40,0.197200,0.228472
50,0.212500,0.213450
60,0.152300,0.189200
70,0.146300,0.191239
80,0.115800,0.202325
90,0.146100,0.190322


You are an expert in emotion classification. Classify the emotions expressed in a Twitter post written in Italian, consisting of an image and a caption-like text. Choose 1-2 emotions from: Love, Anticipation, Disgust, Trust, Joy, Fear, Anger, Surprise, Sadness, Neutral.

Your response must be without any additional text or explanations, and formatted as follows:
Emotions detected: [Emotion(s)]
You are an expert in emotion classification. Classify the emotions expressed in a Twitter post written in Italian, consisting of an image and a caption-like text. Choose 1-2 emotions from: Love, Anticipation, Disgust, Trust, Joy, Fear, Anger, Surprise, Sadness, Neutral.

Your response must be without any additional text or explanations, and formatted as follows:
Emotions detected: [Emotion(s)]
You are an expert in emotion classification. Classify the emotions expressed in a Twitter post written in Italian, consisting of an image and a caption-like text. Choose 1-2 emotions from: Love, Anticipatio

/usr/local/lib/python3.10/dist-packages/torch/utils/checkpoint.py:91: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


You are an expert in emotion classification. Classify the emotions expressed in a Twitter post written in Italian, consisting of an image and a caption-like text. Choose 1-2 emotions from: Love, Anticipation, Disgust, Trust, Joy, Fear, Anger, Surprise, Sadness, Neutral.

Your response must be without any additional text or explanations, and formatted as follows:
Emotions detected: [Emotion(s)]
You are an expert in emotion classification. Classify the emotions expressed in a Twitter post written in Italian, consisting of an image and a caption-like text. Choose 1-2 emotions from: Love, Anticipation, Disgust, Trust, Joy, Fear, Anger, Surprise, Sadness, Neutral.

Your response must be without any additional text or explanations, and formatted as follows:
Emotions detected: [Emotion(s)]
You are an expert in emotion classification. Classify the emotions expressed in a Twitter post written in Italian, consisting of an image and a caption-like text. Choose 1-2 emotions from: Love, Anticipatio

... (13 outputs left out)

You are an expert in emotion classification. Classify the emotions expressed in a Twitter post written in Italian, consisting of an image and a caption-like text. Choose 1-2 emotions from: Love, Anticipation, Disgust, Trust, Joy, Fear, Anger, Surprise, Sadness, Neutral.

Your response must be without any additional text or explanations, and formatted as follows:
Emotions detected: [Emotion(s)]
You are an expert in emotion classification. Classify the emotions expressed in a Twitter post written in Italian, consisting of an image and a caption-like text. Choose 1-2 emotions from: Love, Anticipation, Disgust, Trust, Joy, Fear, Anger, Surprise, Sadness, Neutral.

Your response must be without any additional text or explanations, and formatted as follows:
Emotions detected: [Emotion(s)]
You are an expert in emotion classification. Classify the emotions expressed in a Twitter post written in Italian, consisting of an image and a caption-like text. Choose 1-2 emotions from: Love, Anticipatio

/usr/local/lib/python3.10/dist-packages/torch/utils/checkpoint.py:91: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


You are an expert in emotion classification. Classify the emotions expressed in a Twitter post written in Italian, consisting of an image and a caption-like text. Choose 1-2 emotions from: Love, Anticipation, Disgust, Trust, Joy, Fear, Anger, Surprise, Sadness, Neutral.

Your response must be without any additional text or explanations, and formatted as follows:
Emotions detected: [Emotion(s)]
You are an expert in emotion classification. Classify the emotions expressed in a Twitter post written in Italian, consisting of an image and a caption-like text. Choose 1-2 emotions from: Love, Anticipation, Disgust, Trust, Joy, Fear, Anger, Surprise, Sadness, Neutral.

Your response must be without any additional text or explanations, and formatted as follows:
Emotions detected: [Emotion(s)]
You are an expert in emotion classification. Classify the emotions expressed in a Twitter post written in Italian, consisting of an image and a caption-like text. Choose 1-2 emotions from: Love, Anticipatio

TrainOutput(global_step=98, training_loss=0.42111240114484516, metrics={'train_runtime': 4395.6021, 'train_samples_per_second': 0.176, 'train_steps_per_second': 0.022, 'total_flos': 1.608022259688407e+17, 'train_loss': 0.42111240114484516, 'epoch': 2.0})

# Save model da cambiare il path

In [19]:
trainer.save_model("/source/andrea/new_ft_models/emo_no_correlations_short")

wandb: 
wandb: 🚀 View run pixtral_finetune-2_epochs_3e-05_lr_emotions_no_correlations_short at: 
wandb: Find logs at: wandb/run-20251014_124002-pmtarmzi/logs
